# Введение в MapReduce модель на Python

In [ ]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

In [ ]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

Модель элемента данных

In [ ]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

In [ ]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [ ]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

In [ ]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [ ]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

In [ ]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output) # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [ ]:
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

In [ ]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [ ]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [ ]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных.

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [ ]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*

mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL

In [ ]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication

In [ ]:
from typing import Iterator
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all map tasks

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])

def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, np.float64(1.6724592011493455)),
 (1, np.float64(1.6724592011493455)),
 (2, np.float64(1.6724592011493455)),
 (3, np.float64(1.6724592011493455)),
 (4, np.float64(1.6724592011493455))]

## Inverted index

In [ ]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)

def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)

def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('what', ['0', '1']),
 ('is', ['0', '1', '2']),
 ('it', ['0', '1', '2']),
 ('a', ['2']),
 ('banana', ['2'])]

## WordCount

In [ ]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [ ]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]

def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers

def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)

  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*

flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount

In [ ]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)

  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

# try to set COMBINER=REDUCER and look at the number of values sent over the network
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

56 key-value pairs were sent over a network.


[(0, [('', 6), ('a', 2), ('banana', 2), ('is', 18), ('it', 18)]),
 (1, [('what', 10)])]

## TeraSort

In [ ]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for value in split:
        yield (value, None)

  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])

def MAP(value:int, _):
  yield (value, None)

def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, np.float64(0.01242474408720673)),
   (None, np.float64(0.07928411292407533)),
   (None, np.float64(0.11765675106653872)),
   (None, np.float64(0.11972346618083118)),
   (None, np.float64(0.12056396435862193)),
   (None, np.float64(0.22440698892286592)),
   (None, np.float64(0.23819568363604815)),
   (None, np.float64(0.2530614531466969)),
   (None, np.float64(0.261377229561746)),
   (None, np.float64(0.31620627191234696)),
   (None, np.float64(0.3315720073590598)),
   (None, np.float64(0.3347830793721259)),
   (None, np.float64(0.34453981994508964)),
   (None, np.float64(0.34936704130796004)),
   (None, np.float64(0.4125040143568235)),
   (None, np.float64(0.44204684838279207)),
   (None, np.float64(0.4625969694520964)),
   (None, np.float64(0.46313056130850716))]),
 (1,
  [(None, np.float64(0.5098266681929343)),
   (None, np.float64(0.586160646530552)),
   (None, np.float64(0.6744616135215905)),
   (None, np.float64(0.716442354081398)),
   (None, np.float64(0.7982258712

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [ ]:
import random
from typing import Iterator, NamedTuple

test_numbers = [random.randint(-100, 100) for _ in range(10)]

def RECORDREADER():
    for idx, num in enumerate(test_numbers):
        yield (idx, num)

def MAP(_, value: int):
    yield ("max", value)

def REDUCE(key: str, values: Iterator[int]):
    max_val = max(values)
    yield (key, max_val)

output_max = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(f"Исходные данные: {test_numbers}")
print(f"Максимальное значение: {output_max}")
print(f"Проверка: ожидалось {max(test_numbers)}")

Исходные данные: [-60, -98, 18, 28, 88, -24, -8, 60, -87, -91]
Максимальное значение: [('max', 88)]
Проверка: ожидалось 88


### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [ ]:
def MAP(_, value: int):
    yield ("mean", value)

def REDUCE(key: str, values: Iterator[int]):
    values_list = list(values)
    mean_val = sum(values_list) / len(values_list)
    yield (key, mean_val)

output_mean = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(f"Исходные данные: {test_numbers}")
print(f"Среднее арифметическое: {output_mean}")
print(f"Проверка: ожидалось {sum(test_numbers)/len(test_numbers)}")

Исходные данные: [-60, -98, 18, 28, 88, -24, -8, 60, -87, -91]
Среднее арифметическое: [('mean', -17.4)]
Проверка: ожидалось -17.4


### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [ ]:
def groupbykey(iterable):
    sorted_items = sorted(iterable, key=lambda x: x[0])
    result = []
    current_key = None
    current_values = []

    for key, value in sorted_items:
        if key != current_key:
            if current_key is not None:
                result.append((current_key, current_values))
            current_key = key
            current_values = [value]
        else:
            current_values.append(value)
    if current_key is not None:
        result.append((current_key, current_values))

    return result

test_data = [('a', 1), ('b', 2), ('a', 3), ('c', 4), ('b', 5)]
print("Исходные данные:", test_data)
print("Результат groupbykey:", groupbykey(test_data))

# Сравнение с оригинальной функцией
def groupbykey_original(iterable):
    t = {}
    for (k2, v2) in iterable:
        t[k2] = t.get(k2, []) + [v2]
    return t.items()

print("Результат оригинальной функции:", list(groupbykey_original(test_data)))

Исходные данные: [('a', 1), ('b', 2), ('a', 3), ('c', 4), ('b', 5)]
Результат groupbykey: [('a', [1, 3]), ('b', [2, 5]), ('c', [4])]
Результат оригинальной функции: [('a', [1, 3]), ('b', [2, 5]), ('c', [4])]


### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [ ]:
import random
import numpy as np
from typing import Iterator

test_numbers = [4, 4, 8, 14, 10, 13, 12, 6, 3, 3, 12, 1, 1, 7, 7,
                15, 9, 13, 11, 15, 7, 1, 4, 9, 4, 3, 2, 2, 13, 2]

def INPUTFORMAT():
    global maps

    def RECORDREADER(split):
        for idx, value in enumerate(split):
            yield (idx, value)

    split_size = int(np.ceil(len(test_data) / maps))
    for i in range(0, len(test_numbers), split_size):
        yield RECORDREADER(test_numbers[i:i+split_size])

def MAP(_, value: int):
    yield (value, None)

def COMBINER(key: int, values: Iterator):
    yield (key, None)

def REDUCE(key: int, values: Iterator):
    yield (key, None)

def PARTITIONER(obj):
    global reducers
    return hash(obj) % reducers

maps = 3
reducers = 2

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]

print(f"Исходные данные: {test_numbers}")
print(f"Проверка: ожидалось {np.unique(test_numbers)}")
partitioned_output

30 key-value pairs were sent over a network.
Исходные данные: [4, 4, 8, 14, 10, 13, 12, 6, 3, 3, 12, 1, 1, 7, 7, 15, 9, 13, 11, 15, 7, 1, 4, 9, 4, 3, 2, 2, 13, 2]
Проверка: ожидалось [ 1  2  3  4  6  7  8  9 10 11 12 13 14 15]


[(0,
  [(2, None),
   (4, None),
   (6, None),
   (8, None),
   (10, None),
   (12, None),
   (14, None)]),
 (1,
  [(1, None),
   (3, None),
   (7, None),
   (9, None),
   (11, None),
   (13, None),
   (15, None)])]

#Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [ ]:
class Person(NamedTuple):
    id: int
    name: str
    age: int
    city: str

people = [
    Person(1, "Иван", 25, "Москва"),
    Person(2, "Мария", 30, "СПб"),
    Person(3, "Петр", 25, "Москва"),
    Person(4, "Анна", 35, "Казань"),
    Person(5, "Сергей", 25, "Москва"),
]

def RECORDREADER():
    for person in people:
        yield (person.id, person)

def MAP(_, person: Person):
    if person.city == "Москва" and person.age == 25:
        yield (person.id, person)

def REDUCE(key: int, values: Iterator[Person]):
    for val in values:
        yield (key, val)

output_selection = list(MapReduce(RECORDREADER, MAP, REDUCE))
output_selection

[(1, Person(id=1, name='Иван', age=25, city='Москва')),
 (3, Person(id=3, name='Петр', age=25, city='Москва')),
 (5, Person(id=5, name='Сергей', age=25, city='Москва'))]

### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [ ]:
def RECORDREADER():
    for person in people:
        yield (person.id, person)

def MAP(_, person: Person):
    projected = (person.name, person.age)
    yield (projected, projected)

def REDUCE(key: tuple, values: Iterator[tuple]):
    yield (key, key)

output_projection = list(MapReduce(RECORDREADER, MAP, REDUCE))
print([v for (k, v) in output_projection])

[('Анна', 35), ('Иван', 25), ('Мария', 30), ('Петр', 25), ('Сергей', 25)]


### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [ ]:
set1 = [1, 2, 3, 4, 5]
set2 = [4, 5, 6, 7, 8]

def RECORDREADER():
    for val in set1:
        yield (("set1", val), val)
    for val in set2:
        yield (("set2", val), val)

def MAP(_, value: int):
    yield (value, value)

def REDUCE(key: int, values: Iterator[int]):
    yield (key, key)

output_union = list(MapReduce(RECORDREADER, MAP, REDUCE))
print("Объединение множеств:", [val for _, val in output_union])

Объединение множеств: [1, 2, 3, 4, 5, 6, 7, 8]


### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [ ]:
def RECORDREADER():
    for val in set1:
        yield (("set1", val), val)
    for val in set2:
        yield (("set2", val), val)

def MAP(_, value: int):
    yield (value, value)

def REDUCE(key: int, values: Iterator[int]):
    values_list = list(values)
    if len(values_list) == 2:
        yield (key, key)

output_intersection = list(MapReduce(RECORDREADER, MAP, REDUCE))
print("Пересечение множеств:", [val for _, val in output_intersection])

Пересечение множеств: [4, 5]


### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [ ]:
R = [1, 2, 3, 4, 5]
S = [4, 5, 6, 7, 8]

def REDUCE(key: int, values: Iterator[int]):
    values_list = list(values)
    if key in R and key not in S:
        yield (key, key)

def RECORDREADER():
    for val in R:
        yield ((val, "R"), val)
    for val in S:
        yield ((val, "S"), val)

def MAP(_, value: int):
    yield (value, value)

output_difference = list(MapReduce(RECORDREADER, MAP, REDUCE))
print("Разница R \\ S:", [val for _, val in output_difference])

Разница R \ S: [1, 2, 3]


### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [ ]:
class R(NamedTuple):
    a: int
    b: int

class S(NamedTuple):
    b: int
    c: int

relation_r = [R(a=1, b=2), R(a=2, b=3), R(a=3, b=4)]
relation_s = [S(b=2, c=5), S(b=3, c=6), S(b=4, c=7)]

def RECORDREADER():
    for t in relation_r:
        yield (("R", t.b), t)
    for t in relation_s:
        yield (("S", t.b), t)

def MAP(_, tuple_val):
    if isinstance(tuple_val, R):
        yield (tuple_val.b, ("R", tuple_val.a))
    else:
        yield (tuple_val.b, ("S", tuple_val.c))

def REDUCE(key: int, values: Iterator[tuple]):
    values_list = list(values)
    r_values = [v[1] for v in values_list if v[0] == "R"]
    s_values = [v[1] for v in values_list if v[0] == "S"]

    for a in r_values:
        for c in s_values:
            yield ((a, key, c), (a, key, c))

output_join = list(MapReduce(RECORDREADER, MAP, REDUCE))
print("Natural Join (a,b,c):", [v for (k, v) in output_join])

Natural Join (a,b,c): [(1, 2, 5), (2, 3, 6), (3, 4, 7)]


### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [ ]:
data_group = [
    ("A", 10), ("B", 20), ("A", 15), ("C", 30),
    ("B", 25), ("A", 5), ("C", 35), ("B", 10)
]

def RECORDREADER():
    for idx, (group, val) in enumerate(data_group):
        yield (idx, (group, val))

def MAP(_, value):
    group, val = value
    yield (group, val)

def REDUCE(key: str, values: Iterator[int]):
    total = sum(values)
    yield (key, total)

output_group = list(MapReduce(RECORDREADER, MAP, REDUCE))
print("Группировка и агрегация (сумма):")
for key, val in output_group:
    print(f"  {key}: {val}")

Группировка и агрегация (сумма):
  A: 30
  B: 55
  C: 65


#

### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


In [ ]:
import numpy as np

matrix = np.random.rand(100, 50)
vector = np.random.rand(50)

def RECORDREADER():
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            yield ((i, j), matrix[i, j])

def MAP(coordinates, value):
    i, j = coordinates
    yield (i, value * vector[j])

def REDUCE(i, values):
    total = sum(values)
    yield (i, total)

output_matvec = list(MapReduce(RECORDREADER, MAP, REDUCE))
result = np.array([val for _, val in sorted(output_matvec, key=lambda x: x[0])])
reference = np.dot(matrix, vector)

print(np.allclose(result, reference))

True


## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$.





In [ ]:
# MapReduce model
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [ ]:
import numpy as np
I = 2
J = 3
K = 40
small_mat = np.random.rand(I, J) # it is legal to access this from RECORDREADER, MAP, REDUCE
big_mat = np.random.rand(J, K)

def RECORDREADER():
    for j in range(big_mat.shape[0]):
        for k in range(big_mat.shape[1]):
            yield ((j, k), big_mat[j, k])

def MAP(k1, v1):
    j, k = k1
    w = v1
    # solution code that yield(k2,v2) pairs
    for i in range(small_mat.shape[0]):
        yield ((i, k), small_mat[i, j] * w)

def REDUCE(key, values):
    i, k = key
    # solution code that yield(k3,v3) pairs
    yield ((i, k), sum(values))

Проверьте своё решение

In [ ]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat)
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution)) # should return true

True

In [ ]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max(i for ((i,k), vw) in reduce_output)

1

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [ ]:
import numpy as np
from typing import Iterator

I, J, K = 3, 4, 5
M = np.random.rand(I, J)
N = np.random.rand(J, K)

def RECORDREADER():
    for i in range(I):
        for j in range(J):
            yield (("M", i, j), M[i, j])

    for j in range(J):
        for k in range(K):
            yield (("N", j, k), N[j, k])

def MAP(key, value):
    matrix_type = key[0]

    if matrix_type == "M":
        _, i, j = key
        for k in range(K):
            yield ((i, k), (j, value, "M"))
    else:
        _, j, k = key
        for i in range(I):
            yield ((i, k), (j, value, "N"))

def REDUCE(key, values):
    i, k = key

    m_dict = {}
    n_dict = {}

    for val in values:
        j, val_val, matrix_type = val
        if matrix_type == "M":
            m_dict[j] = val_val
        else:
            n_dict[j] = val_val

    total = 0
    for j in range(J):
        if j in m_dict and j in n_dict:
            total += m_dict[j] * n_dict[j]

    yield ((i, k), total)

reference_solution = np.matmul(M, N)
solution = MapReduce(RECORDREADER, MAP, REDUCE)
print(np.allclose(reference_solution, asmatrix(solution)))

True


Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER.

In [ ]:
I, J, K = 6, 5, 4
matrix1 = np.random.rand(I, J)
matrix2 = np.random.rand(J, K)
solution =  np.matmul(matrix1, matrix2)

maps = 3
reducers = 2

def INPUTFORMAT():
    global maps

    all_elements = []
    for i in range(I):
        for j in range(J):
            all_elements.append((("M", i, j), matrix1[i, j]))
    for j in range(J):
        for k in range(K):
            all_elements.append((("N", j, k), matrix2[j, k]))

    split_size = int(np.ceil(len(all_elements) / maps))

    def RECORDREADER(split_elements):
        for key, value in split_elements:
            yield (key, value)

    for i in range(0, len(all_elements), split_size):
        yield RECORDREADER(all_elements[i:i+split_size])

def MAP(key, value):
    if key[0] == "M":
        _, i, j = key
        yield (j, ("M", i, value))
    else:
        _, j, k = key
        yield (j, ("N", k, value))

def REDUCE(j, values):
    m_values = []
    n_values = []

    for val in values:
        if val[0] == "M":
            _, i, m_val = val
            m_values.append((i, m_val))
        else:
            _, k, n_val = val
            n_values.append((k, n_val))

    partial = {}
    for i, m_val in m_values:
        for k, n_val in n_values:
            key = (i, k)
            partial[key] = partial.get(key, 0) + m_val * n_val

    for key, val in partial.items():
        yield (key, val)

partitioned_output = list(MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None))

result = np.zeros_like(solution)
for _, partition in partitioned_output:
    for key, value in partition:
        result[key] += value

print(np.allclose(solution, result))

50 key-value pairs were sent over a network.
True


Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы? (Полагаю, что нет)

In [ ]:
def INPUTFORMAT():
    global maps

    all_elements = []
    for i in range(I):
        for j in range(J):
            all_elements.append((("M", i, j), matrix1[i, j]))
    for j in range(J):
        for k in range(K):
            all_elements.append((("N", j, k), matrix2[j, k]))

    np.random.shuffle(all_elements)

    split_size = max(1, int(np.ceil(len(all_elements) / maps)))

    def RECORDREADER(split_elements):
        for key, value in split_elements:
            yield (key, value)

    for i in range(0, len(all_elements), split_size):
        if i < len(all_elements):
            yield RECORDREADER(all_elements[i:i+split_size])

def MAP(key, value):
    if key[0] == "M":
        _, i, j = key
        yield (j, ("M", i, value))
    else:
        _, j, k = key
        yield (j, ("N", k, value))

def REDUCE(j, values):
    m_values = []
    n_values = []

    for val in values:
        if val[0] == "M":
            _, i, m_val = val
            m_values.append((i, m_val))
        else:
            _, k, n_val = val
            n_values.append((k, n_val))

    partial = {}
    for i, m_val in m_values:
        for k, n_val in n_values:
            key = (i, k)
            partial[key] = partial.get(key, 0) + m_val * n_val

    for key, val in partial.items():
        yield (key, val)

partitioned_output = list(MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None))

result = np.zeros_like(solution)
for _, partition in partitioned_output:
    for key, value in partition:
        result[key] += value

print(np.allclose(solution, result))

50 key-value pairs were sent over a network.
True


In [ ]:
def INPUTFORMAT():
    global maps

    all_elements = []
    for i in range(I):
        for j in range(J):
          ### Добавляем вероятность создания элемента
            if np.random.random() > 0.3:
                all_elements.append((("M", i, j), matrix1[i, j]))
    for j in range(J):
        for k in range(K):
          ### Добавляем вероятность создания элемента
            if np.random.random() > 0.3:
                all_elements.append((("N", j, k), matrix2[j, k]))

    np.random.shuffle(all_elements)

    split_size = max(1, int(np.ceil(len(all_elements) / maps)))

    def RECORDREADER(split_elements):
        for key, value in split_elements:
            yield (key, value)

    for i in range(0, len(all_elements), split_size):
        if i < len(all_elements):
            yield RECORDREADER(all_elements[i:i+split_size])

def MAP(key, value):
    if key[0] == "M":
        _, i, j = key
        yield (j, ("M", i, value))
    else:
        _, j, k = key
        yield (j, ("N", k, value))

partitioned_output = list(MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None))

result = np.zeros_like(solution)
for _, partition in partitioned_output:
    for key, value in partition:
        result[key] += value

print(np.allclose(solution, result))

28 key-value pairs were sent over a network.
False
